# Lección 2: DEEP LEARNING
## A. Arquitectura óptima para clasificación de imágenes

Para la clasificación de imágenes, la arquitectura estándar y más eficiente es la **Red Neuronal Convolucional (CNN)**. A diferencia de las redes densas (ANN) que vimos antes, las CNN están diseñadas para procesar datos con estructura de rejilla, como los píxeles de una imagen.

### Componentes clave de una CNN:
1. **Capas Convolucionales (Conv2D):** Funcionan como "filtros" que recorren la imagen buscando patrones específicos como bordes, texturas o formas.
2. **Capas de Agrupación (Pooling):** Reducen el tamaño de la imagen (downsampling) para conservar solo la información más relevante y acelerar el procesamiento.
3. **Capas Densas (Fully Connected):** Se ubican al final de la red para tomar todas las características extraídas por las convoluciones y realizar la clasificación final.

Esta arquitectura es "óptima" porque posee **invariancia espacial**: el modelo puede reconocer un objeto sin importar en qué parte de la imagen se encuentre.

---

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Definimos una arquitectura CNN clásica (tipo LeNet/AlexNet simplificada)
# Usamos Input + Reshape para mantener consistencia con el resto del proyecto
model_cnn = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Reshape((28, 28, 1)),  # Adaptamos (28,28) -> (28,28,1) para Conv2D

    # Primera capa convolucional: busca 32 filtros de 3x3
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),  # Reduce la imagen a la mitad

    # Segunda capa convolucional: extrae patrones más complejos
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Transformación de matriz a vector plano
    layers.Flatten(),

    # Clasificador final
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 categorías de ropa Fashion-MNIST
])

# Compilación
model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("Arquitectura de la CNN creada:")
model_cnn.summary()

Arquitectura de la CNN creada:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 13, 13, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 5, 5, 64)         0         
 2D)                                                             
                                                                 
 flatten (Flatten)       

### Análisis de la arquitectura propuesta

* **Jerarquía de filtros:** Las primeras capas suelen detectar líneas simples, mientras que las capas más profundas combinan esas líneas para identificar formas complejas (como ojos, ruedas o rostros).
* **Reducción de parámetros:** Gracias al *Pooling*, logramos que la red sea mucho más ligera. Sin estas capas, el número de conexiones en las capas densas sería tan grande que el modelo sería extremadamente lento y propenso al sobreajuste.
* **Salida Softmax:** A diferencia de la *Sigmoide* (usada en clasificación binaria), aquí usamos **Softmax** porque nos permite obtener una distribución de probabilidad sobre las 10 categorías de ropa, asegurando que la suma de todas las predicciones sea igual a 1.

## B. Justificación: Red Densa (MLP) vs. Red Convolutiva (CNN)

Al trabajar con imágenes, la elección de la arquitectura no es arbitraria. Aunque una **Red Densa (Multi-Layer Perceptron)** puede procesar píxeles, las **Redes Convolucionales (CNN)** son técnica y lógicamente superiores para datos con estructura espacial.

### Diferencias fundamentales:
* **Red Densa:** Trata cada píxel como una variable independiente. Para una imagen de 200x200 píxeles a color, la primera capa tendría 120,000 conexiones por cada neurona, lo que provoca una explosión de parámetros y pérdida de la relación espacial (si mueves un objeto un píxel a la derecha, la red densa lo ve como algo totalmente nuevo).
* **Red Convolutiva:** Utiliza **filtros compartidos** que recorren la imagen. Esto permite la **invariancia espacial** (reconoce el patrón sin importar su ubicación) y reduce drásticamente el número de pesos que la red debe aprender, enfocándose en jerarquías de características (bordes -> formas -> objetos).

---

In [2]:
# Demostración de la explosión de parámetros: Densa vs Convolutiva
import tensorflow as tf
from tensorflow.keras import layers, models

# Supongamos una imagen pequeña de 64x64 píxeles a color (RGB)
input_shape = (28, 28, 1)

# 1. Modelo Denso (MLP)
model_denso = models.Sequential([
    layers.Flatten(input_shape=input_shape),
    layers.Dense(512, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# 2. Modelo Convolutivo (CNN)
model_cnn_comparacion = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

print(f"Parámetros en el modelo DENSO:     {model_denso.count_params():,}")
print(f"Parámetros en el modelo CNN:       {model_cnn_comparacion.count_params():,}")
print(f"\nReducción de parámetros: {model_denso.count_params() / model_cnn_comparacion.count_params():.1f}x menos con CNN")

Parámetros en el modelo DENSO:     407,050
Parámetros en el modelo CNN:       54,410

Reducción de parámetros: 7.5x menos con CNN


### Análisis de la elección

* **Eficiencia de Parámetros:** Como se observa en el código, el modelo denso genera millones de parámetros incluso para una imagen pequeña, lo que facilita el **sobreajuste (overfitting)** y requiere muchísima memoria. La CNN, con menos parámetros, logra mejores resultados al ser más específica.
* **Conectividad Local:** La CNN "mira" píxeles vecinos gracias a los kernels (filtros), capturando la esencia de la imagen. La red densa ignora que un píxel está al lado de otro, perdiendo la información de texturas y contornos.
* **Veredicto:** Para clasificación de imágenes, la arquitectura convolutiva es la única opción viable en términos computacionales y de precisión.

## C. Selección del Framework: Keras/TensorFlow vs. PyTorch

En la industria y la investigación existen dos gigantes que dominan el desarrollo de Deep Learning. La elección depende totalmente del objetivo del proyecto:

* **Keras (TensorFlow):** Es el estándar para quienes buscan velocidad de desarrollo y despliegue sencillo. Es extremadamente amigable (High-Level API), lo que permite construir modelos potentes con muy pocas líneas de código. Es ideal para producción y para quienes se están iniciando.
* **PyTorch:** Es el preferido en el mundo académico y de investigación. Su enfoque es más "pythonico" y permite un control total sobre cada operación matemática del modelo. Es excelente si necesitas crear arquitecturas experimentales muy complejas desde cero.

**Nuestra elección:** Para este trabajo utilizaremos **Keras**, ya que permite prototipar rápidamente arquitecturas convolucionales y densas sin perder el foco en la lógica del modelo.

---

In [3]:
# Verificación del framework seleccionado y sus capacidades
import tensorflow as tf
import keras

# Versiones
print(f"Versión de TensorFlow: {tf.__version__}")
print(f"Versión de Keras:       {keras.__version__}")

# GPU
dispositivos = tf.config.list_physical_devices('GPU')
if dispositivos:
    print(f"\n¡GPU detectada! El entrenamiento será rápido:")
    for d in dispositivos:
        print(f"  → {d.name}")
else:
    print("\nEntrenando en CPU (suficiente para modelos pequeños).")

# Modelo de prueba mínimo
print("\nEstructura mínima de un modelo en Keras:")
test_model = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(1)
])
test_model.compile(optimizer='sgd', loss='mse')
test_model.summary()
print("\nModelo compilado con éxito ✓")

Versión de TensorFlow: 2.12.0
Versión de Keras:       2.12.0

Entrenando en CPU (suficiente para modelos pequeños).

Estructura mínima de un modelo en Keras:
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_5 (Dense)             (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________

Modelo compilado con éxito ✓


### Análisis de la elección

* **Curva de Aprendizaje:** Keras reduce la "fricción" técnica. Mientras que en PyTorch tendríamos que escribir manualmente el bucle de entrenamiento (for loops, optim.step, etc.), Keras lo resuelve internamente con el método `.fit()`.
* **Ecosistema:** Al usar Keras dentro de TensorFlow, tenemos acceso a herramientas como **TensorBoard** para visualizar el entrenamiento en tiempo real, lo cual es fundamental para diagnosticar problemas de sobreajuste.
* **Conclusión:** Para los objetivos de esta clase, Keras nos ofrece el equilibrio perfecto entre potencia y simplicidad, permitiéndonos centrar el esfuerzo en entender las capas y no en la sintaxis compleja del framework.